In [ ]:
# ==============================
# FULL ASSIGNMENT 2 PIPELINE (SINGLE CELL)
# ==============================

# INSTALL (run once)
# !pip install torch torchaudio librosa numpy matplotlib jiwer dtw-python soundfile g2p_en coqui-tts openai-whisper

import os
import numpy as np
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from g2p_en import G2p
from jiwer import wer
from dtw import dtw
import whisper
from TTS.api import TTS

# ==============================
# CONFIG
# ==============================
SAMPLE_RATE = 16000
FRAME_SIZE = 0.025
HOP_SIZE = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# LOAD AUDIO
# ==============================
def load_audio(path):
    audio, sr = librosa.load(path, sr=SAMPLE_RATE)
    return audio

def denoise(audio):
    return librosa.effects.preemphasis(audio)

# ==============================
# MFCC FEATURE
# ==============================
def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(
        y=audio, sr=SAMPLE_RATE,
        n_mfcc=13,
        n_fft=int(SAMPLE_RATE * FRAME_SIZE),
        hop_length=int(SAMPLE_RATE * HOP_SIZE)
    )
    return mfcc.T

# ==============================
# LID MODEL
# ==============================
class LIDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(13, 64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)

def train_lid(X, y):
    model = LIDModel().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    X = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    y = torch.tensor(y, dtype=torch.long).to(DEVICE)

    for epoch in range(3):
        opt.zero_grad()
        out = model(X)
        loss = loss_fn(out.view(-1, 2), y.view(-1))
        loss.backward()
        opt.step()
        print(f"LID Epoch {epoch} Loss:", loss.item())

    return model

# ==============================
# WHISPER + CONSTRAINT
# ==============================
whisper_model = whisper.load_model("base")

def constrained_decode(audio_path, vocab):
    result = whisper_model.transcribe(audio_path)
    words = result["text"].split()

    # simple constraint (boost domain words)
    new_words = []
    for w in words:
        if w.lower() in vocab:
            new_words.append(w.upper())
        else:
            new_words.append(w)

    return " ".join(new_words)

# ==============================
# IPA MAPPING
# ==============================
g2p = G2p()

def hinglish_to_ipa(text):
    return " ".join(["".join(g2p(w)) for w in text.split()])

# ==============================
# TRANSLATION (SIMPLE DICT)
# ==============================
translation_dict = {
    "hello": "namaskar",
    "speech": "bhashan",
    "model": "model",
    "learning": "sikhai",
    "language": "bhasha"
}

def translate(text):
    return " ".join([translation_dict.get(w.lower(), w) for w in text.split()])

# ==============================
# TTS (VOICE CLONING)
# ==============================
tts = TTS(model_name="tts_models/multilingual/multi-dataset/your_tts")

def generate_tts(text, speaker_wav, output="output.wav"):
    tts.tts_to_file(
        text=text,
        speaker_wav=speaker_wav,
        file_path=output
    )

# ==============================
# PROSODY (DTW)
# ==============================
def extract_f0(audio):
    f0, _, _ = librosa.pyin(audio, fmin=50, fmax=300)
    return np.nan_to_num(f0)

def apply_dtw(f0_src, f0_tgt):
    alignment = dtw(f0_src, f0_tgt)
    return alignment

# ==============================
# ANTI-SPOOF MODEL
# ==============================
class SpoofDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(13, 32, 3)
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.mean(dim=2)
        return self.fc(x)

# ==============================
# FGSM ATTACK
# ==============================
def fgsm_attack(model, x, y, eps=0.01):
    x.requires_grad = True
    out = model(x)
    loss = nn.CrossEntropyLoss()(out, y)
    model.zero_grad()
    loss.backward()
    return x + eps * x.grad.sign()

# ==============================
# MAIN PIPELINE
# ==============================

# INPUT FILES (IMPORTANT)
INPUT_AUDIO = "original_segment.wav"
SPEAKER_REF = "student_voice_ref.wav"

audio = load_audio(INPUT_AUDIO)
audio = denoise(audio)

# LID
mfcc = extract_mfcc(audio)
X = np.expand_dims(mfcc, axis=0)
y = np.zeros((1, mfcc.shape[0]))  # dummy labels (replace with real)

lid_model = train_lid(X, y)

# STT
vocab = ["stochastic", "cepstrum", "acoustic"]
text = constrained_decode(INPUT_AUDIO, vocab)
print("\nTranscript:\n", text)

# IPA
ipa = hinglish_to_ipa(text)
print("\nIPA:\n", ipa)

# TRANSLATE
translated = translate(text)
print("\nTranslated:\n", translated)

# TTS
generate_tts(translated, SPEAKER_REF)

# PROSODY
f0_src = extract_f0(audio)
f0_tgt = extract_f0(load_audio("output.wav"))
alignment = apply_dtw(f0_src, f0_tgt)
print("\nDTW Alignment Done")

# METRICS (example)
ref = "this is reference text"
print("\nWER:", wer(ref, text))

print("\nPIPELINE COMPLETE")